# 05 Top Routes and Stations

Build top route and station tables and export them for the Streamlit Top Routes & Stations page.

Exports:
- top_routes_city_year.csv
- top_stations_city_year.csv

In [1]:
import sys
import pandas as pd

sys.path.insert(0, '..')
from utils import load_app_ready, export_df

In [2]:
df = load_app_ready()
df.shape

(15907082, 22)

In [3]:
required = [
    'city_name', 'year', 'trip_id',
    'start_station_name', 'end_station_name'
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns for top-routes analysis: {missing}')

base = df.dropna(subset=['city_name', 'year', 'start_station_name', 'end_station_name']).copy()
base['year'] = pd.to_numeric(base['year'], errors='coerce')
base = base.dropna(subset=['year'])
base['year'] = base['year'].astype(int)

top_routes = (
    base.groupby(['city_name', 'year', 'start_station_name', 'end_station_name'], as_index=False)
    .agg(trips=('trip_id', 'count'))
    .sort_values(['city_name', 'year', 'trips'], ascending=[True, True, False])
)

departures = (
    base.groupby(['city_name', 'year', 'start_station_name'], as_index=False)
    .agg(departures=('trip_id', 'count'))
    .rename(columns={'start_station_name': 'station_name'})
)
arrivals = (
    base.groupby(['city_name', 'year', 'end_station_name'], as_index=False)
    .agg(arrivals=('trip_id', 'count'))
    .rename(columns={'end_station_name': 'station_name'})
)

top_stations = departures.merge(
    arrivals,
    on=['city_name', 'year', 'station_name'],
    how='outer'
)
top_stations['departures'] = top_stations['departures'].fillna(0).astype(int)
top_stations['arrivals'] = top_stations['arrivals'].fillna(0).astype(int)
top_stations['total_trips'] = top_stations['departures'] + top_stations['arrivals']
top_stations = top_stations.sort_values(
    ['city_name', 'year', 'total_trips'],
    ascending=[True, True, False]
)

top_routes.head(), top_stations.head()

(     city_name  year start_station_name end_station_name  trips
 692     Bergen  2018    Møllendalsplass  Nonneseterplass   1305
 762     Bergen  2018    Nonneseterplass  Møllendalsplass   1221
 804     Bergen  2018           Nykirken   Småstrandgaten    945
 1159    Bergen  2018          Tårnplass         Nykirken    780
 1021    Bergen  2018      Solheimsviken       Media City    661,
    city_name  year     station_name  departures  arrivals  total_trips
 30    Bergen  2018   Småstrandgaten        6641      7005        13646
 21    Bergen  2018  Møllendalsplass        6841      6677        13518
 4     Bergen  2018   Cornerteateret        6319      6635        12954
 31    Bergen  2018    Solheimsviken        6459      6418        12877
 35    Bergen  2018        Tårnplass        6349      6405        12754)

In [4]:
export_df('top_routes_city_year', top_routes)
export_df('top_stations_city_year', top_stations)
print('Exported top routes and stations tables.')

[bridge] Exported DataFrame: top_routes_city_year.csv (target: NOTEBOOK_EXPORTS_PATH)
[bridge] Exported DataFrame: top_stations_city_year.csv (target: NOTEBOOK_EXPORTS_PATH)
Exported top routes and stations tables.
